### RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [3]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\vijay\AppData\Local\Temp\ipykernel_12348\3494988372.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [ ]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all pdfs in a directory"""
    all_documents=[]
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files=list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")

        try:
            loader=PyPDFLoader(str(pdf_file))
            documents=loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata["source_file"]=pdf_file.name
                doc.metadata["file_type"]='pdf'

            all_documents.extend(documents)

        except Exception as e:
            raise ValueError(e) 

    print(f"\nTotal Documents Loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents= process_all_pdfs("../data")


### Text splitting get into chunks

In [ ]:
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""

    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs=text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk
    if split_docs:
        print("\nExample Chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [ ]:
chunks=split_documents(all_pdf_documents)
chunks